In [1]:
!pip install pyspark --quiet

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SupplyChainAnalysis") \
    .getOrCreate()

print("Spark session started:", spark.version)

Spark session started: 4.0.2


In [2]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("supplier_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("order_date", StringType(), True),
    StructField("delivery_date", StringType(), True)
])

spark_df = spark.read.csv(
    '/content/orders_01.csv',
    header=True,
    schema=schema
)


In [3]:
print("Row count:", spark_df.count())
print("Columns:", spark_df.columns)
spark_df.printSchema()
spark_df.show()

Row count: 15
Columns: ['order_id', 'supplier_id', 'product_id', 'quantity', 'order_date', 'delivery_date']
root
 |-- order_id: integer (nullable = true)
 |-- supplier_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- order_date: string (nullable = true)
 |-- delivery_date: string (nullable = true)

+--------+-----------+----------+--------+----------+-------------+
|order_id|supplier_id|product_id|quantity|order_date|delivery_date|
+--------+-----------+----------+--------+----------+-------------+
|       1|          1|         2|      50|2025-06-01|   2025-06-06|
|       2|          2|         5|      30|2025-06-05|   06/08/2025|
|       3|          3|         1|     100|2025-06-07|   2025-06-17|
|       4|          4|         6|      20|2025-06-10|         NULL|
|       5|          5|         3|      75|06/11/2025|   2025-06-15|
|       6|          1|         7|      15|2025-06-12|   2025-06-17|
|       7|        

In [18]:
from pyspark.sql.functions import udf, col, datediff, current_date, when
from pyspark.sql.types import StringType
from datetime import datetime
def normalize_date(date_str):
    if date_str is None or date_str.strip() == "":
        return None
    formats_to_try = ["%Y-%m-%d", "%m/%d/%Y", "%Y/%m/%d"]
    for fmt in formats_to_try:
        try:
            return datetime.strptime(date_str.strip(), fmt).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return None

normalize_date_udf = udf(normalize_date, StringType())

# apply to both date columns
spark_df = spark_df.withColumn("order_date_clean", normalize_date_udf(col("order_date")))
spark_df = spark_df.withColumn("delivery_date_clean", normalize_date_udf(col("delivery_date")))

# now cast the cleaned strings to actual date type
spark_df = spark_df.withColumn("order_date_parsed", col("order_date_clean").cast("date"))
spark_df = spark_df.withColumn("delivery_date_parsed", col("delivery_date_clean").cast("date"))

# calculate delay_days
spark_df = spark_df.withColumn(
    "delay_days",
    when(
        col("delivery_date_parsed").isNotNull(),
        datediff(current_date(), col("delivery_date_parsed"))
    ).otherwise(
        datediff(current_date(), col("order_date_parsed"))
    )
)

# flag delayed orders
spark_df = spark_df.withColumn(
    "is_delayed",
    when(col("delay_days") > 0, 1).otherwise(0)
)

# filter only delayed shipments
delayed_df = spark_df.filter(col("is_delayed") == 1)

print("Total orders:", spark_df.count())
print("Delayed orders:", delayed_df.count())
delayed_df.select("order_id", "supplier_id", "delivery_date", "delay_days", "is_delayed").show()

Total orders: 15
Delayed orders: 15
+--------+-----------+-------------+----------+----------+
|order_id|supplier_id|delivery_date|delay_days|is_delayed|
+--------+-----------+-------------+----------+----------+
|       1|          1|   2025-06-06|       381|         1|
|       2|          2|   06/08/2025|       379|         1|
|       3|          3|   2025-06-17|       370|         1|
|       4|          4|         NULL|       377|         1|
|       5|          5|   2025-06-15|       372|         1|
|       6|          1|   2025-06-17|       370|         1|
|       7|          2|   2025-06-16|       371|         1|
|       8|       NULL|   2025-06-19|       368|         1|
|       9|          3|   2025/06/20|       367|         1|
|      10|          4|         NULL|       371|         1|
|      11|          5|   2025-06-22|       365|         1|
|      12|          1|   2025-06-23|       364|         1|
|      13|          2|   2025-06-21|       366|         1|
|      14|          

In [21]:
from pyspark.sql.functions import count

delayed_by_supplier = delayed_df.groupBy("supplier_id") \
    .agg(count("order_id").alias("delayed_order_count")) \
    .orderBy("delayed_order_count", ascending=False)

print("--- Delayed Orders Grouped by Supplier ---")
delayed_by_supplier.show()

--- Delayed Orders Grouped by Supplier ---
+-----------+-------------------+
|supplier_id|delayed_order_count|
+-----------+-------------------+
|          1|                  3|
|          3|                  3|
|          2|                  3|
|       NULL|                  2|
|          5|                  2|
|          4|                  2|
+-----------+-------------------+



In [22]:
delayed_by_supplier.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("/content/delayed_by_supplier_csv")

In [23]:
delayed_by_supplier.write \
    .mode("overwrite") \
    .parquet("/content/delayed_by_supplier_parquet")

print("Saved successfully!")
print("CSV folder: /content/delayed_by_supplier_csv")
print("Parquet folder: /content/delayed_by_supplier_parquet")

Saved successfully!
CSV folder: /content/delayed_by_supplier_csv
Parquet folder: /content/delayed_by_supplier_parquet


In [24]:
df_check = spark.read.parquet("/content/delayed_by_supplier_parquet")
df_check.show()

+-----------+-------------------+
|supplier_id|delayed_order_count|
+-----------+-------------------+
|          1|                  3|
|          3|                  3|
|          2|                  3|
|       NULL|                  2|
|          5|                  2|
|          4|                  2|
+-----------+-------------------+

